<a href="https://colab.research.google.com/github/lsteffenel/CHPS0906/blob/main/TP4/20_PINNs_Thermals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing PINNs on thermal diffusion problems
PINNs are neural networks made to approximate solutions of PDE systems. In a few words, they leverage experimental data and physical equation to efficiently approximate the solution of PDE systems in a continuous manner. On a physician perspective PINNs can be viewed as postulating the form of the solution of a PDE and then try to find the parameters that would allow to fit to the solution and boundary conditions.


The aim of PINNs is to approximate physical equations and leverage experimental data. Let $f(x,t)$ be a neural network. $f$ is trained to minimize a loss on a dataset of measurements $\{u(x_i,t_i)\}_{i \in [n]}$ (with, ideally, boundary conditions in it)
\begin{equation}
\text{MSE}_u= \frac{1}{n} \sum_{i=1}^n \|f(x_i,t_i) - u(x_i,t_i)\|^2 \tag{1}
\end{equation}

On the other hand the physical system is bound to follow certain rules in the form of Partial Derivative Equation. Generally the can be written in the form of a differential operator $\mathcal{N}$
\begin{equation}
\frac{\partial u}{\partial t}+ \mathcal{N}[u]=0 \tag{2}
\end{equation}
For instance for the heat equation:
\begin{equation*}
\mathcal{N}[u]= - D(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2})
\end{equation*}

In order to infuse the neural network with physical knowledge we add to the statistical error a physical error, that is to say on a set of collocation points $\{(x_k,t_k)\}_{k\in[N]}$ that are usually different from the measurement points and that can change thoughout the training procedure we compute a physical error:
\begin{equation}
\text{MSE}_{\phi}=\frac{1}{N} \sum_{k=1}^N \| \frac{\partial f}{\partial t}|_{(x_k,t_k)} + \mathcal{N}[f]|_{(x_k,t_k)} \|^2 \tag{3}
\end{equation}

Thus the final error to be optimized is
\begin{equation*}
\text{MSE}= \text{MSE}_u + \text{MSE}_{\phi}
\end{equation*}

Partial derivative can then be computed leveraging automatic differentiation implemented in numerous deep learning libraries such as tensorflow, pytorch, jax


In [ ]:

import torch
import torch.nn as nn
import torch.autograd as autograd
import torch.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

class PINN(nn.Module):
    def __init__(self, input_size):
        """
        Simple 5 layer neural network
        """
        super(PINN,self).__init__()
        self.linear1 = nn.Linear(input_size, 500)
        self.linear2 = nn.Linear(500,500)
        self.linear3 = nn.Linear(500,500)
        self.linear4 = nn.Linear(500,500)
        self.linear5 = nn.Linear(500,1)
        self.nonlin = nn.Sigmoid() #Non Linearity is chosen to be sigmoid since we differenciate twice

    def forward(self,x):
        return self.nonlin(self.linear5(self.nonlin(self.linear4(self.nonlin(self.linear3(self.nonlin(self.linear2(self.nonlin(self.linear1(x))))))))))


Our illustrating case will be the following one: heat diffusion in a homogeneous bar where the bar is at temperature 1 on the right side and temperature 0 on the other (this case can be made from arbitrary temperature thanks to an appropriate scaling)

In [ ]:
def plot_case():
    xs,ys,zs = np.meshgrid(np.arange(0,2,0.001),np.arange(0,1,0.001),[0])
    zs[xs<1]=1.
    plt.contourf(xs.squeeze(),ys.squeeze(),zs.squeeze(),levels=[0.,0.1,0.2,0.3,0.4,0.45,0.50,0.55,0.6,0.7,0.8,0.9,1.])
    plt.colorbar()
    plt.show()
    plt.clf()
plot_case()

First we define the boundary loss $\text{MSE}_u$, we devised that it would have 3 components (other choices can be made, for instance in case of measurements):
* The initial conditions as displayed in the chart above: $\forall (x,y) \in \Omega,T(x,y,t=0)=T_0(x,y)$
* The border condition of a null thermal flux: $\forall (x,y) \in \partial\Omega, \vec{\nabla}_{(x,y)}T.\vec{n}(x,y)=0$ where $\vec{n}(x,y)$ is the normal vector to the boundary
* The final condition as a homogeneous state:  $\forall (x,y) \in \Omega,T(x,y,t=\infty)=T_f$


In [ ]:
def boundary_loss(model,N,device=None,dtype=None):
   # State is composed of x,y and t
   # Sampling randomly initial state, thus for t=0:
   values_init = torch.rand(N,3)*np.array([2.,1.,0.])
   label_init = torch.zeros(N,1)
   label_init[values_init[:,[0]]<1]=1. # left side temperature =1 when x<0

   #setting to device and dtype
   values_init=values_init.to(device=device,dtype=dtype)
   label_init = label_init.to(device=device,dtype=dtype)

   #Error initial condition
   loss_init = torch.mean((model(values_init)-label_init)**2)

   #Sampling final  thus for t=1
   values_end = torch.rand(N,3)*torch.tensor([2.,1.,0.,])
   values_end[:,2]=torch.ones_like(values_end[:,2]) # setting t=1
   label_end = torch.ones(N,1)*0.5 #uniform temperature at the end

   #setting to device and dtype
   values_end=values_end.to(device=device,dtype=dtype)
   label_end = label_end.to(device=device,dtype=dtype)

   #Error end condition
   loss_end = torch.mean((model(values_end)-label_end)**2)

   #Sampling the border of the rectangle (perimeter 2+1+2+1=6)
   values_limit=torch.rand(N)*6

   #Initializing the border at random positions and time
   values_border = torch.zeros(N,3)

   #first edge: x=random(0,2), y=0
   values_border[:,0][values_limit<2]=values_limit[values_limit<2]
   values_border[:,1][values_limit<2]=0. # explicit is better than implicit
   #second edge: x=2, y=random(0,1)
   values_border[:,1][(2<values_limit)*(values_limit<3)]=values_limit[(2<values_limit)*(values_limit<3)] - 2.
   values_border[:,0][(2<values_limit)*(values_limit<3)]=2.
   #third edge: x=random(0,2), y=1
   values_border[:,0][(3<values_limit)*(values_limit<5)]=values_limit[(3<values_limit)*(values_limit<5)]-3.
   values_border[:,1][(3<values_limit)*(values_limit<5)]=1.

   #fourth edge: x=0, y=random(0,1)
   values_border[:,1][5<values_limit]=values_limit[5<values_limit]-5.
   values_border[:,0][5<values_limit]=0 #explicit is better than implicit

   # at random times
   values_border[:,2]= torch.rand(values_border.shape[0])
   #setting to device and dtype
   values_border=values_border.to(device=device,dtype=dtype)
   values_border.requires_grad=True

   # Computing grads with autodifferentiation
   u = model(values_border)
   ux = autograd.grad(u,values_border,create_graph=True,grad_outputs=torch.ones_like(u),allow_unused = True,retain_graph=True)[0]

   #first edge: u_y = 0
   grads_border= torch.sum(ux[:,1][values_limit<2]**2)
   #first edge: u_x = 0
   grads_border += torch.sum(ux[:,0][(2<values_limit)*(values_limit<3)]**2)
   #first edge: u_y = 0
   grads_border+= torch.sum(ux[:,1][(3<values_limit)*(values_limit<5)]**2)
   #first edge: u_x = 0
   grads_border+= torch.sum(ux[:,0][5<values_limit]**2)
   #averaging
   grads_border/=N

   # Aggregating losses
   loss_boundary_condition=loss_end + loss_init + grads_border
   return loss_boundary_condition

Now we need to implement the physical loss in order to enforce the dynamics (equation (3))
The real time evolving from 0 to $\infty$ we determined (by studying real dynamics) that the final state is reached closely at $t=800$ thus the time for the neural network will be $t'=t/800$ for it to be between 0 and 1.
For the physical loss, the time derivatives are computed with respect to $t$ not $t'$

In [ ]:
def physical_loss(model,N,device=None,dtype=None):
    # on random space points
    space = torch.rand(N,2)*torch.tensor([2.,1.])
    space = space.to(device=device,dtype=dtype)
    space.requires_grad=True

    # reparametrization of the time (the real time evolves from 0 to 800 (almost final state), and the time for the neural net evolves from 0 to 1)
    time = torch.rand(N,1)*600
    time = time.to(device=device,dtype=dtype)
    time.requires_grad=True
    # reparametrization
    t_s =time/600
    # concatenante space and reparametrized time (=real time / 800)
    u = model(torch.cat((space,t_s),axis=1))

    # Computing derivatives
    # in real time
    ut =  autograd.grad(u,time,create_graph=True,grad_outputs=torch.ones_like(u),allow_unused = True,retain_graph=True)[0]
    # first order in space
    ux = autograd.grad(u,space,create_graph=True,grad_outputs=torch.ones_like(u),allow_unused = True,retain_graph=True)[0]
    # second order in space
    uxx = autograd.grad(ux,space,create_graph=True,grad_outputs=torch.ones_like(ux),allow_unused = True,retain_graph=True)[0]
    # heat diffusion equation with diffusion coefficient D=0.001
    philoss = (ut[:,0] - 0.001*(uxx[:,0]+uxx[:,1]))**2
    return torch.sum(philoss)

All remains to code the training loop

In [ ]:
def train(model,boundary_loss,physical_loss,epochs=10000,lr=0.001):


    optimizer = torch.optim.Adam(model.parameters(),lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, epochs//3, gamma=0.1)
    # Parameters (chosen arbitrarily)
    N_boundary=20000
    N_phi = 10000


    # logs
    log_freq=500
    logs=[]

    for epoch in range(epochs):
        loss =  boundary_loss(model,N_boundary,device=device,dtype=dtype)+physical_loss(model,N_phi,device=device,dtype=dtype)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        logs.append(loss.item())
        if (epoch+1)%log_freq==0:
            print("Epoch {},".format(epoch), "Loss: ",np.mean(logs))
            logs=[]
    return model

We add some utilities functions so as to plot the solution at a given timestep given the model

In [ ]:
def plot_timestep(model,t,device=None,dtype=None):
    #building grid of points
    xs,ys,ts = np.meshgrid(np.arange(0,2,0.002),np.arange(0,1,0.002),[t])

    #computing solution for the given time step
    with torch.no_grad():
        X_grid= torch.tensor(np.stack([xs,ys,ts],axis=-1)).to(device=device,dtype=dtype)
        Z_pt = torch.clamp(model(X_grid).squeeze(),0,1)
        Z_mesh = Z_pt.cpu().numpy()
    #plot contours
    plt.contourf(xs.squeeze(),ys.squeeze(),Z_mesh,levels=np.linspace(0.,1.,100))
    plt.colorbar()
    plt.show()
    plt.clf()

Performing the training and plotting results

In [ ]:
1!wget "https://github.com/lsteffenel/CHPS0906/raw/refs/heads/main/TP4/model_simple_initial_condition.pt"

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.float
    # Model and optimization
    model = PINN(3)
    model.to(device=device,dtype=dtype)
    model.train()
    model.load_state_dict(torch.load("model_simple_initial_condition.pt",map_location=device))
    model = train(model,boundary_loss,physical_loss,epochs=1000,lr=0.00001)
    torch.save(model.state_dict(),"model_simple_initial_condition_finetuned.pt")
else:
    model = PINN(3)
    model.load_state_dict(torch.load("model_simple_initial_condition.pt",map_location=torch.device('cpu')))

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dtype = torch.float
for timestep in np.linspace(0,600,5,endpoint=True):
        plot_timestep(model,timestep/600,device=device,dtype=dtype)

We can test for other boundary conditions

In [ ]:
def boundary_loss_circle(model,N,device=None,dtype=None):
   # State is composed of x,y and t
   # Sampling randomly initial state, thus for t=0:
   values_init = torch.rand(N,3)*np.array([2.,1.,0.])
   label_init = torch.zeros(N,1)
   label_init[((values_init[:,[0]]-1.)**2 +(values_init[:,[1]]-0.5)**2)<0.25]=1. # temperature=1 in a disk of radius 0.5 centered in (1,0.5)

   #setting to device and dtype
   values_init=values_init.to(device=device,dtype=dtype)
   label_init = label_init.to(device=device,dtype=dtype)

   #Error initial condition
   loss_init = torch.mean((model(values_init)-label_init)**2)

   #Sampling final  thus for t=1
   values_end = torch.rand(N,3)*np.array([2.,1.,0.,])
   values_end[:,2]=torch.ones_like(values_end[:,2]) # setting t=1
   label_end = torch.ones(N,1)*math.pi*0.25/2 #uniform temperature at the end equal to \piR^2/ (2*1)

   #setting to device and dtype
   values_end=values_end.to(device=device,dtype=dtype)
   label_end = label_end.to(device=device,dtype=dtype)

   #Error end condition
   loss_end = torch.mean((model(values_end)-label_end)**2)

   #Sampling the border of the rectangle (perimeter 2+1+2+1=6)
   values_limit=torch.rand(N)*6

   #Initializing the border at random positions and time
   values_border = torch.rand(N,3)

   #first edge: x=random(0,2), y=0
   values_border[:,0][values_limit<2]=values_limit[values_limit<2]
   values_border[:,1][values_limit<2]=0.
   #second edge: x=2, y=random(0,1)
   values_border[:,1][(2<values_limit)*(values_limit<3)]=values_limit[(2<values_limit)*(values_limit<3)] - 2.
   values_border[:,0][(2<values_limit)*(values_limit<3)]=2.
   #third edge: x=random(0,2), y=1
   values_border[:,0][(3<values_limit)*(values_limit<5)]=values_limit[(3<values_limit)*(values_limit<5)]-3.
   values_border[:,1][(3<values_limit)*(values_limit<5)]=1.

   #fourth edge: x=0, y=random(0,1)
   values_border[:,1][5<values_limit]=values_limit[5<values_limit]-5.
   values_border[:,0][5<values_limit]=0.

   # Setting random times
   values_border[:,2]=torch.rand(N)
   #setting to device and dtype
   values_border=values_border.to(device=device,dtype=dtype)
   values_border.requires_grad=True

   # Computing grads with autodifferentiation
   u = model(values_border)
   ux = autograd.grad(u,values_border,create_graph=True,grad_outputs=torch.ones_like(u),allow_unused = True,retain_graph=True)[0]

   #first edge: u_y = 0
   grads_border= torch.sum(ux[:,1][values_limit<2]**2)
   #first edge: u_x = 0
   grads_border += torch.sum(ux[:,0][(2<values_limit)*(values_limit<3)]**2)
   #first edge: u_y = 0
   grads_border+= torch.sum(ux[:,1][(3<values_limit)*(values_limit<5)]**2)
   #first edge: u_x = 0
   grads_border+= torch.sum(ux[:,0][5<values_limit]**2)
   #averaging
   grads_border/=N

   # Aggregating losses
   loss_boundary_condition=loss_end + loss_init + grads_border
   return loss_boundary_condition

In [ ]:
!wget "https://github.com/lsteffenel/CHPS0906/raw/refs/heads/main/TP4/model_circle_initial_condition.pt"

In [ ]:
# you have cuda it trains, otherwise it loads
if torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.float
    # Model and optimization
    model2 = PINN(3)
    model2.to(device=device,dtype=dtype)
    model2.train()
    model2.load_state_dict(torch.load("model_circle_initial_condition.pt",map_location=device))
    model2 = train(model2,boundary_loss_circle,physical_loss,epochs=1000,lr=0.00001)
    torch.save(model2.state_dict(),"model_circle_initial_condition_finetuned.pt")
else:
    model2 = PINN(3)
    model2.load_state_dict(torch.load("model_circle_initial_condition.pt",map_location=torch.device('cpu')))

In [ ]:
for timestep in np.linspace(0,100,5,endpoint=True):
        #rescaling time for NN
        plot_timestep(model2,timestep/600,device=device,dtype=dtype)

We note that the prediction appears to be asymmetrical, this pattern is reproduced every time we perform the experiment. Something that is not expected on the real solution presented below on the same timesteps

The results are still remarkable indeed the neural network does not have data in between $t=0$ and $t=1$ which means that is is learning completely in a unsupervised manner the dynamics of the system.